# RAG para el tutor de matemáticas

### Sesión 7 · Módulo 3 — Buscar primero, responder después

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

### El hueco que dejó el scorecard de M2

M2 midió el tutor con un eval set de tres bloques y encontró dos fallos de
naturaleza **distinta**:

| Bloque | Qué falla | Naturaleza del problema |
|---|---|---|
| **Cálculo** | El modelo se equivoca en problemas de varios pasos | **Habilidad** |
| **Conocimiento** | No sabe qué porcentaje vale el examen final del colegio | **Conocimiento** |

El segundo no se arregla entrenando. Ese dato **no existe en los pesos de
ningún modelo del mundo**: es de una institución concreta, y además puede
cambiar el año que viene. Reentrenar por cada cambio de reglamento es
absurdo; indexar el documento nuevo toma minutos.

Esa es la pregunta correcta de S07, y conviene tenerla explícita:

> **¿Mi problema es de habilidad o de conocimiento?**
> Si la respuesta cambia con el tiempo o vive en SUS documentos → RAG.
> Si es un patrón estable de comportamiento → fine-tuning.

M1 (fine-tuning) enseñó **forma**: el formato pasó del 12% al 100%.
M3 (RAG) debe aportar **conocimiento consultable**.

### La adaptación que este dominio exige

El RAG de clase responde preguntas sobre documentos. Nuestro tutor tiene que
hacer **dos cosas distintas** según lo que le pregunten:

- *"¿Cuánto vale el examen final?"* → responder **solo** desde el contexto, y
  citar la fuente. Si no está, admitirlo.
- *"¿Cuánto se paga con 20% y luego 10% de descuento sobre 200000?"* →
  **resolverlo**, usando el contexto solo si contiene una fórmula o estrategia
  útil.

Un prompt que diga "responde SOLO con base en el contexto" rompería el segundo
caso: el modelo se negaría a calcular porque el resultado no está escrito en
ningún documento. La sección 8 construye un prompt que distingue ambos modos, y
es la pieza de diseño propia de este notebook.

### La hipótesis que vamos a poner a prueba

RAG debería **ganar en el bloque de conocimiento** y **no aportar (o incluso
estorbar) en el de cálculo**, porque meter párrafos irrelevantes en el prompt
añade ruido a un problema que solo requiere razonar. Lo mediremos: si el bloque
de cálculo baja, hay que reportarlo, no esconderlo.

## 2 · Objetivos

1. Construir el pipeline RAG completo **sin frameworks**: ingest → chunk →
   embed → store → retrieve → augment → generate.
2. Comparar **búsqueda por palabras clave contra búsqueda semántica** sobre
   consultas reales del dominio.
3. Diseñar un prompt aumentado con **válvula de escape** que funcione tanto
   para datos institucionales como para problemas de cálculo.
4. Correr **el harness de M2 sin modificar una línea** sobre el sistema RAG.
5. Comparar `scorecard_rag` contra `scorecard_m2` **por bloque**, y diagnosticar
   los fallos que queden: ¿fue la búsqueda o fue la generación?

## 0 · Preparación del entorno

> **Activen la GPU (T4).** Se cargan el generador, el juez y el modelo de
> embeddings.

In [ ]:
%pip install -q "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" \
    sentence-transformers chromadb "scikit-learn>=1.3"
%pip uninstall -y -q torchao
print("Librerías instaladas. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

In [ ]:
import os, random, re
import numpy as np
import torch
import transformers

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | {DEVICE}")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

## 3 · El eval set

El eval set **no es el corpus de entrenamiento**. Es un conjunto nuevo, escrito
a mano, que cumple los cuatro criterios de S05:

| Criterio | Cómo se cumple aquí |
|---|---|
| **Representativo** | Problemas que un estudiante real plantearía, con enunciados en lenguaje natural. |
| **Con salida esperada** | Cada caso trae la respuesta correcta y un `criterio` explícito de qué la hace correcta. |
| **Cubre casos difíciles** | Es el eje del diseño: multipaso, distractores, porcentajes encadenados, proporcionalidad inversa, redondeo contextual. |
| **No contaminado** | Ningún caso proviene de `math_tutor_dataset.jsonl`. El generador lo verifica automáticamente y falla si hay solapamiento. |

**Por qué hizo falta uno nuevo.** En M1 el modelo *sin entrenar* ya resolvía el
90.91% del conjunto de validación. Un eval set con ese nivel de dificultad no
puede discriminar entre sistemas: todo se ve bien. Este está construido para
que el modelo falle, porque un eval set que no reta al sistema no informa nada.

### Los tres bloques

- **A · CÁLCULO (12 casos)** — mide **habilidad**. Cada uno tiene un modo de
  fallo documentado en su `criterio` (por ejemplo, sumar descuentos sucesivos
  en lugar de encadenarlos).
- **B · CONOCIMIENTO (4 casos)** — mide **conocimiento institucional** que
  ningún modelo puede tener en sus pesos. El baseline debería abstenerse aquí;
  si en cambio inventa una cifra con seguridad, es una alucinación y el
  scorecard la registra. **Este bloque es el que motiva el RAG de M3.**
- **C · ADVERSARIAL (5 casos)** — mide **robustez**, siguiendo el red-teaming
  de S06: premisa falsa, indefinición matemática, datos insuficientes, fuera de
  dominio y mantenimiento del rol.

Supera de largo el mínimo de M2 (≥10 gold + ≥2 adversariales): 16 gold y 5
adversariales.

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_EVAL = Path("data/eval_set_m2.jsonl")
SHA_EVAL  = "ba9c4f3b2d80d687b4a56e1d93e260a964e0b375e61de1d27d10c41a43e1f96c"

_BLOB_EVAL = (
    "H4sIAAAAAAAC/61aTW8bSZK9L7D/ISFgABlL0/wURS4aC60tzwpwWx6pe4HB7EBIslJ0eouV7CqWIE1jfsUe9uyjDz4M+jBA"
    "XwYY/rF5LzLri0W5qR4D3bCYlR+RES8jXkTmj0c2Opqpo8jePu/1jzrqaB67H3LDtoWOF3ns2Jjl841dO7au8hh/6szdmGSh"
    "I5PoSLrYZJ1v2OH7RG10HJtULdxqnWo1VAv9QWcqMmrj0sTGscuUVv3Tca+n1ibDL0ykVZ5o9aAGKnXSA90Xeh4bdB2MJuOd"
    "rq6r3umlxhqJ6mOiYqqu+vvfXubbj8nGqcgmJnUKU0TmLjfxnUn+g6KabG1Sig1p32Enqj9Tb3QWxFzkJtvoBGJv/z8I+Y0a"
    "j/Fv938S6T5Ad0gS5Cz6D9jfS/qNGk1r/YczdR5j71CLMpmfS/2b74O+/R7lLzqPKAsFdnG+sNufEg4JO3we+nKBMOYKmxEB"
    "1K1NdDzzH7wuuNdFajcmtbLXl7neQB+Ou19Yl2Di0oY6m0GryqSpS5VNMGZlIuvw552OLVRuYpWaDMaH4rqiRXzIZR7M/eNR"
    "gY8kX2G9hWACPVzKRhHq6M9//td/+bEOuMEhgLPJrUtXstBNZLMNZAeMdANz5wmFz3RMfb3XD2o4gNo2eWQ1toKt9U9VBqSs"
    "8g8GuwDM+iNpeO9WczR0aaAwfGNNYtRIAS0wq84Ek9BxutHod209AqvZVWIWJrMbAXoObaaJyzqA4cLDMKtaVVb1hq4Tj4lH"
    "MQmZIg0cpyaGsrkULRarZPvzisD2iIpMc6vDQQ2n38ppje1Cr7D8GraNGwLJIfMHiiMJ4SHQNT3ZB63pSTVyF1uQ1SR5srDY"
    "A0CziPMHI9JnyqblBjJ1XKi1U+i0g3N6ZzOBOgXMzL17JgZZucjEOMZmbpRdJjB6CumfDL7pSRt5w0OQB99yZ9KMwMsTnIHI"
    "ZG1Pl2CKgBk6sIVeQyz0pWoH3bFaGZw5qHz78xyCEUF6jt7QEgf2/ffiszI/5BaC0+n1eY5jy9EdYJC9Yfuw4IN4Pr/stNav"
    "hjrfpG51TLCtNTwxvHJCHT6KuJey440lWOL6ZnSYbiZ7omP0bmgwLhevoe6KqJE5/C5r0s78kOciNXzfyX4vxvYwbwtp93Zp"
    "1CJImqrCNMoDDGoXB5d2vRgAlE1q3QXsSs+zPP0VjoxytdE0OgRNqYlcEhl3Q1WY+02u4x00wasgYJpMe8jD0yUZTgT30IfW"
    "fHzqqpc8sDjQcmJinixGWqdW24/3sBx8Xdm1jIWZDBCXX/dBjwLhFY5kRBjM/NLbv8q036iT7nSk9Dp1WApyrHCaTc3ybwHh"
    "ezhpfxhu4at9DBM/IxJ3lM62nwUWQSOClPcwgFY6Te1cz5p+clLKvg8p1dc9LimoGjFvHluwBV2umar/Ont5cabOrq4u/vNM"
    "sLLmh1SdKOd3CV3BkTl4LwScJyNl0obJ+BCYeGfIWGcWUPEuRnCkNuQzaoJz+L82dstU9CcKvt1+ouqAIr3QjC9zF2eeefW6"
    "w0ljQEG5KpCgY+iPOdxcjusBCBGv5LlNsknFSYnacSxdGV9k4pkIDSiJMI9GKb2ah1glTkY8F9wij0HQCcPchG4Ec3Fb8EN7"
    "vcigF3a0C41XZcTxMudJMTX8MQKN982zBukhHr49+/3lVeGKaamInrxT+LkMJFNc022eRDZ9MmYGe3zLySGggbIWjKsfzOOs"
    "XEuUgi3cLRCN4NT7Dc0DzwkGS7r80AElmnsLwg1BZDru1MzhS7hJKKmPMTqy3A1URROt9dIRaeYWp8TeOWFI1QQutUvagsob"
    "9CqaXotURNuaVB69xG6PQu6shIgsYKHBSvxZMT+CU697WgSXBmsnyTPLPBFgFtvG8lom/tLuZ2EyP/tUZh+NHqHg/st+Dv5G"
    "zmpYGyctp5e7w19vLylJlq90UvdG/ZEse1wQA7JAiWBDmGL7cwIIPQu00HP3Rbz9mKH16aFNpG4DcHIIAL2nF68V3+DIbJro"
    "OwP6YvOw/UkNXoyJOmwgtnMQH8gd54mn5C+G/BTOEv6P9FzyjpVONyYrkOXHeeozpJq3HwmxGqTgwYrGsGo5yZe4tpfDDziG"
    "nM+EDXuaMqjWqeHpd5QxdYn0ei69vpE4udvXZ4BehLBC2K2hv8WQmYyjP2OQ3Qurk2reFqp0AHY95lbAxsm6Or/+7rKjEscf"
    "4q4bKCPEgkTYSxNRhbqejKiTPWg6PcidpY4eTbwMyd2NFSLezPmABoRqnFih1y5BZoicQxLBVZ6KU+oPVLT9rJu8WFrAo1OY"
    "Dn8malrOAsbF407/wEhgM7ApaHnlvgQbP8hV+f2J8OMB7DgJ6z/3C9SQA6JdrTtjR5h+iiGnfsQ++4dPu7Z/t6MsdfH2v8+v"
    "rs9mJIRZsUZHAlvYfTe40hT+bRlrqcwwLUbMghfX5MhImB/+SQZ02rb+9BDrL41jUmT1DQtIooJW2gVkYjtKxE2WeYytrCwz"
    "rl6RcAl/GIRfSCcTYaQmgx/Fv6wRACgfgILt50TSWpqWmhgVM9AXSRvPCE8OAxVOvCnQhghfZ9hFphfmyrwLA7OwPGYWpazM"
    "k5ly+Jdgtf2YGl2m+HG5ZQmkJcIGo15r3WbA8/NwhmKzmGHECUYSIh8dPqzyuJms85y9seRgtJdrDUatqXah+touc9l/MOuM"
    "yQKTOcEB+R1Gi8CMB6mkb7F+esYPWVrY6/eeSKS8z2nGsfOSG/gYBoe+/cxplM6ZCsGvkyWNfwPx9XvHvXolTQd7K5QoCGwY"
    "5IrUNQ7zPO5w4PXug3cOknAs9Jt2yGVdAj6BHsRHt09mLCvXIPGKs37wVr0vvkt+5/uf9h5hNae9R0nNu1JrwflczuSEMY4L"
    "S7aphB5vUcoOFdXjz+R0wAgUZBECN/5KtEbEbqOhf2AckirojRdzl1MHn7LSVISWypqvfENw4WoUf+hpNrxsgrNMp4ot3TNx"
    "9orl70kP2vg+qRcVkX/ld6gGdwdSJyymRWUTKZSf2c9SwElCT5b7VX6JR6OoYDzXqkSeyWKi/KHwmO7gpIabclCx8ExEkf4T"
    "KQV1R+Oa77gGk/X1A8yDajc/o9ewO+nvQxfb90Q2X4R+d/n21fnV2atAYBiSt582REMNQ0NgRh0XJkPVZ7WOzbN/MoaJWC3w"
    "DA4EzxylPk9gUIMCLUZiHus/tRMzVmuqet6YSavm/cIH8YNDpf+Ux0KAkSAgs0/ET0qnjiAwoja3n3zxC3ykIzUvuUOADyP2"
    "fLakG2CR5kpGDuaBRfotJQAsIxJ84cpk7/hqNU4ic8itx4vTGpp+r0VE1rq4jyJUjsKmWd8o6+O1ilFc8VxGsReTGuAaRYSZ"
    "OsaCwuCP0e2ZlAhejBm/xi/6ewMY23cReH3xVpVmk3JAlT8mbmWZZqcsQM2trmNxMH5xMlLHvJwqh38lUi1i1hGJRdpXdy5x"
    "KGdYRoRWhENZFUq6qS3aoNVmibRStso63krj+GQbu8lD0s+cJYz0hT2We2KztI4s+wfAUFVBtO752HNNxUJ7UuLecYQtnJ3v"
    "OMpi0KjpUWvzdmieJ0jfbdWFeN3SGMGsEwIwXZcIjbosaseoboEbmH+X68ers9/6D0goRLgHtbAhzkVu4QP6IXa+cTd+7sIo"
    "1f1Zr23yweEmjzUT8xsNYtWw9d//hhAmJlsK+wXBxdVf6iJUJZCg4yCaRV5cFUJ1oeZSdC5Uzfn5uaCaEhGRfBBmWR0hLRNf"
    "75+j88tr1wXFf77ZIRu4c1/bqmBZaZi6btqGxF8yb+HYb3QM36FbVl7E+g7V65n6w5FfhY1+Q5Itn0rDadcd/bEFguHBIOBZ"
    "uYHHQpFzBwP1gCAnaoUsAd2kzD+H7khualakEoo6C1PPTXkDE7Gmv8j9/XJxFStVQn8+W/Z/84UVMd+w2wtFuTH+epDpGPp+"
    "YdXaim0sMLHz15J1QMhVuMTSjOEoLYJDVzL1xgn35ekCCOjsYfSVEIAts2XYkX8kK1/nkiogi98DgNHhXiB1GwcTgs0i1i7d"
    "MkSWGhJ+R1dgeMW+gaGpBqRqvmgs+UaYoAEGHFLGw7zBXkMJ3BPMfeXOfc6+mr+2bijJpHImseyt3P6rVRnsWXT2McCnQ2Xt"
    "UB1LGfgBdT2DGjWrts8KIyPp8mUPoomPWTJmT+i5lEsGKcP6q/IqlmUtLJ1XmhKVbn9astpW+HzcezT9jCmvtJlbokO6/YhC"
    "E4SDL8kzVgh2fE/lbArdfCWYeTWysaFJNpQq4Y9dpbKNavX/TpuA1NFdi4mgjZU7HOi4hUeDCpu+wQX1TmXv/N6XLddyEQNI"
    "TukMgLDi4QNjQdBEDUJT5csjux1nSkpbvPHhcyJ/4zPscHrui1U3eeszlBysMZTewO0b3geutp9CjbDj718HHc46xvXhpAWV"
    "d1fn315cn6nXZ2+uz/xrk8BPxM4vL6+uzn97ceXJs+hFUh03R7kJJz9m4nir56kU7YjgDzm81y1+VkTmMFTsAMGrTDbLL5UG"
    "80oLzQ91/QhedjUr/ksUKn/dhwZac1hNVaRkR38UmLn3dm7Fjn84mj4iw/QxEaaNPWS2+RO5Q/m7jdfBoXi18GcgonL5dSMM"
    "R5Pg7AmmdNfyaEvuOvcFvWjn/pHO3aN3A5fg14lQHkvK6/TEJkKVir0TudXRhQOmenso737CungbJymGI55cFRiT4t6k+ZKr"
    "DtQLZNqvL95evLzY/t/bLqpFJQPiii0R0UMAWcp13OtAgA48HDtsfNaD7zjlT3Je+2AqZN+vW+KoLkxoDD+9YWo/SmWGH2U5"
    "N5SoQvOOhhqtGcxlo9qBWDsP/doUzGszebznvW71GybYg/YAE3gh8aq7V86+EXRliQcjfBY03tcLrW1oDw+FthAiFDyzHB7F"
    "SrBqVdxRfFZ3loUKVDN7fEmw/alWamexs17ZhL5Wa+evWSS0gi2QPLFEBUKwsUmbDb7mUyW5iydhl1J5WqTUs1CAh0UZGhi9"
    "LZ9d8tXIe01oMNvyE3vSVu/SkUcKXqTM8NqnNtzbTsTkWZz22qTx7LvLa5Q2r79/jYNx/va78+twNBD3rU/0KDKPiLy3Eudd"
    "BHlfoimW+/VnQGausOs1UWDeLmqwplIXEh1CW+1CTVr82xrBMR7eBKG9Mw9itoHaxtfoUHyBAKWaGWjEmolt0067/UT+CJ3P"
    "cUrUSyhB6e1f/CUMDg+YR9SmjJkES2hqo4MnkIUkA7UgVqiULfDgLHNy17PJSRx3slN4SrgA1BQf8oi3s0K/mITEcO+yelVt"
    "hFWRKC3NnAW2cElFJHnXhKuVjJ1a4Hn9PUqX6tW5enX5LRzrZUAOOS64uhH0xNvPK+tLsjgfQfIGIy0PgucaWugDqfaSSSsy"
    "2i/A6lEW2FSJhDM5+HtVxQ+FfgtY1XQnXLLUeuggQd4LLpAgm5JB0ru5aGOtA9A3PhR9eKd9wxfNy5QJY5Np4qHCA0GUyavU"
    "kMf4opORjAY6cL6Ag0i3MoEQ1r0vrwwnpDunQ193NZ6++tJFd0+i0/Dcw2mv32G66gjaWz6hDY++pRgL2ZAGhfsewCUKyd2s"
    "XJSPwf2f8qh8UrzhHU5O2NAf9fkDq/DFvK/cMiPDHRzkC49piqqmTwtCR5RpM1/E95mQlFs2kh3AlWipSLfr+NNeq45/dfmm"
    "y5eBQnVjbABOkY8TqMyQ3wT6wBUbOgi6TFv7l2I4XNd6o/01E4hPPXGHJeQxpddiZOQu3CSZ2f7F/HoXzLzIp0U1UWQ2kZN/"
    "eYuJLy20/Aia/wFK/EPUfDEAAA=="
)

if not RUTA_EVAL.exists():
    RUTA_EVAL.parent.mkdir(parents=True, exist_ok=True)
    RUTA_EVAL.write_bytes(gzip.decompress(base64.b64decode(_BLOB_EVAL)))
    print(f"Eval set escrito en {RUTA_EVAL} (copia embebida).")
else:
    print(f"Se usará el eval set existente en {RUTA_EVAL}.")

eval_set = [json.loads(l) for l in RUTA_EVAL.read_text(encoding="utf-8").splitlines()]

sha_real = hashlib.sha256(RUTA_EVAL.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "… coincide:", sha_real == SHA_EVAL)
print()

from collections import Counter
print(f"Casos: {len(eval_set)}  ->  {dict(Counter(c['bloque'] for c in eval_set))}")
print()
for c in eval_set:
    print(f"  {c['id']:8s} {c['bloque']:13s} {c['subtipo']:26s} {c['input'][:52]}...")

In [ ]:
# Un caso completo, para ver la estructura.
print(json.dumps(eval_set[5], ensure_ascii=False, indent=2))

---
## 4 · Métricas y embeddings

Las mismas funciones de M1 y M2, sin tocar. El modelo de embeddings cumple aquí
**doble función**: métrica de similitud del harness y motor de búsqueda del
RAG. Es el mismo `paraphrase-multilingual-MiniLM-L12-v2` de S05, S06 y S07.

In [ ]:
import re
import unicodedata
from fractions import Fraction

MARCADOR_RESPUESTA = "Respuesta final:"

_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)
_PAT_PASO1     = re.compile(r"Paso\s*1\s*:", re.IGNORECASE)
_PAT_CATEGORIA = re.compile(r"Categor[ií]a\s*:\s*([a-zA-Z_]+)", re.IGNORECASE)
# La alternativa de fracción va primero: en "3/5" queremos capturar la fracción
# completa, no el "3" suelto.
_PAT_VALOR = re.compile(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?|-?\d+(?:\.\d+)?")


def _sin_miles(texto):
    """Quita la coma como separador de miles. El corpus usa el punto como
    separador decimal y nunca la coma, así que la conversión no es ambigua."""
    return texto.replace(",", "")


def a_float(valor):
    if valor is None:
        return None
    try:
        return float(Fraction(valor)) if "/" in valor else float(valor)
    except (ValueError, ZeroDivisionError):
        return None


def valor_predicho(texto):
    """Extrae la respuesta del modelo en forma canónica.

    Prioridad 1: el número que sigue al marcador 'Respuesta final:'.
    Prioridad 2: el último número del texto.

    El segundo caso importa para que la comparación sea JUSTA: un modelo sin
    fine-tuning no conoce nuestro formato, y penalizarlo por eso mediría
    obediencia al formato, no capacidad matemática. Con el fallback medimos lo
    segundo; el apego al formato se mide aparte con `formato_valido`.
    """
    m = _PAT_RESPUESTA.search(texto)
    if m:
        linea = m.group(1).splitlines()[0]
        v = _PAT_VALOR.search(_sin_miles(linea))
        if v:
            return v.group(0).replace(" ", "")
    todos = _PAT_VALOR.findall(_sin_miles(texto))
    return todos[-1].replace(" ", "") if todos else None


def respuesta_correcta(generado, valor_oro, tol=1e-6):
    a = a_float(valor_predicho(generado))
    b = a_float(valor_oro)
    if a is None or b is None:
        return False
    return abs(a - b) <= tol * max(1.0, abs(b))


def formato_valido(texto):
    """¿El modelo produjo la estructura que le enseñamos?"""
    return bool(_PAT_PASO1.search(texto)) and bool(_PAT_RESPUESTA.search(texto))


def categoria_predicha(texto):
    m = _PAT_CATEGORIA.search(texto)
    return m.group(1).lower() if m else None


def _tokens(texto):
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", t)


def _lcs(a, b):
    """Longitud de la subsecuencia común más larga (programación dinámica)."""
    previa = [0] * (len(b) + 1)
    for x in a:
        actual = [0]
        for j, y in enumerate(b):
            actual.append(previa[j] + 1 if x == y else max(previa[j + 1], actual[j]))
        previa = actual
    return previa[-1]


def rouge_l(generado, referencia):
    """ROUGE-L (F1 sobre la subsecuencia común más larga).

    Se implementa a mano en lugar de usar `evaluate` para que el notebook no
    dependa de descargas en tiempo de ejecución y el número sea exactamente
    reproducible.
    """
    p, r = _tokens(generado), _tokens(referencia)
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)


def evaluar_generacion(generados, registros):
    """Métricas de generación sobre un conjunto de ejemplos.

    exactitud       : ¿la respuesta final es numéricamente correcta?  <- la que importa
    formato_valido  : ¿respetó la estructura Paso N / Respuesta final?
    rouge_l         : ¿se parece el procedimiento al de referencia?
    long_media      : longitud media en palabras (detecta divagación)
    """
    assert len(generados) == len(registros)
    n = len(generados)
    correctas = [respuesta_correcta(g, r["valor"]) for g, r in zip(generados, registros)]
    formatos  = [formato_valido(g) for g in generados]
    rouges    = [rouge_l(g, r["salida"]) for g, r in zip(generados, registros)]
    return {
        "n": n,
        "exactitud": sum(correctas) / n,
        "formato_valido": sum(formatos) / n,
        "rouge_l": sum(rouges) / n,
        "long_media": sum(len(g.split()) for g in generados) / n,
        "_correctas": correctas,
    }


def tabla_metricas(antes, despues, titulo="Baseline vs Fine-tuned"):
    """Imprime la comparación en el formato que usaremos en el informe."""
    filas = [
        ("Exactitud de la respuesta", "exactitud", "{:.1%}"),
        ("Formato válido",            "formato_valido", "{:.1%}"),
        ("ROUGE-L del procedimiento", "rouge_l", "{:.3f}"),
        ("Longitud media (palabras)", "long_media", "{:.1f}"),
    ]
    ancho = 30
    print(titulo)
    print("=" * 68)
    print(f"{'Métrica':{ancho}s} {'Baseline':>12s} {'Fine-tuned':>12s} {'Δ':>10s}")
    print("-" * 68)
    for etiqueta, clave, fmt in filas:
        a, d = antes[clave], despues[clave]
        print(f"{etiqueta:{ancho}s} {fmt.format(a):>12s} {fmt.format(d):>12s} "
              f"{d - a:>+10.3f}")
    print("=" * 68)

### Dimensión 1 · Métricas clásicas

Dos métricas automáticas, y la segunda existe para demostrar por qué **no
basta**.

**1a · Exactitud de la respuesta final** — la métrica del dominio, heredada de
M1: se extrae el número que sigue a `Respuesta final:` y se compara
numéricamente con el esperado. Es objetiva y no admite discusión.

**1b · Similitud por embeddings** — la métrica clásica del lab de S05/S06.
La incluimos porque la entrega la pide y porque es útil para juzgar la
*explicación*, pero la sección siguiente muestra que en matemáticas es
peligrosa si se usa sola.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# El MISMO modelo de embeddings de S05, S06 y S07 (multilingüe, pequeño).
st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


def sim_embeddings(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

#### Por qué la similitud sola no sirve para un tutor de matemáticas

Esta demostración es el equivalente, en nuestro dominio, del Lab A de S05
—donde BLEU premiaba la respuesta equivocada—. Aquí el que falla es el
embedding: dos respuestas idénticas salvo por el número obtienen una similitud
altísima, aunque una esté bien y la otra mal.

Es la razón por la que la Dimensión 3 usa un criterio numérico y no el
`sim >= 0.60` de la plantilla de clase.

In [ ]:
referencia = ("Paso 1: Aplicamos el primer descuento: 200000 × 0.80 = 160000.\n"
              "Paso 2: El segundo se aplica sobre el precio ya rebajado: 160000 × 0.90 = 144000.\n"
              "Respuesta final: 144000 pesos")

candidatos = {
    "correcta (misma redacción)": referencia,
    "CORRECTA pero parafraseada": ("Primero quitamos el 20 por ciento, quedando 160000 pesos. "
                                   "Después restamos el 10 por ciento de esa cantidad. "
                                   "El precio final es de 144000 pesos."),
    "INCORRECTA (sumó los %)":    ("Paso 1: Sumamos los descuentos: 20% + 10% = 30%.\n"
                                   "Paso 2: 200000 × 0.70 = 140000.\n"
                                   "Respuesta final: 140000 pesos"),
}

print(f"{'candidato':<30} {'similitud':>10} {'¿correcta?':>12}")
print("-" * 54)
for nombre, texto in candidatos.items():
    s = sim_embeddings(texto, referencia)
    ok = respuesta_correcta(texto, "144000")
    print(f"{nombre:<30} {s:>10.3f} {str(ok):>12}")
print("-" * 54)
print("La respuesta INCORRECTA obtiene una similitud altísima: comparte")
print("estructura, vocabulario y casi todos los números. Con el umbral de 0.60")
print("del lab de clase, contaría como acierto. Por eso el criterio de dominio")
print("tiene que mirar el número, no el parecido del texto.")

---
## 5 · El corpus documental

Seis documentos que cubren los dos tipos de conocimiento que al tutor le
faltan:

| Documento | Tipo | Por qué está |
|---|---|---|
| SIEE (sistema de evaluación) | **Institucional** | Porcentajes de nota, nota mínima, recuperaciones. Datos que ningún modelo puede saber. |
| Plan de área de matemáticas | **Institucional** | Qué se enseña en cada grado. |
| Protocolo de errores frecuentes | **Pedagógico** | Estrategias por tipo de error, incluidas las trampas del eval set. |
| Formulario de geometría | **De referencia** | Fórmulas de áreas, perímetros y volúmenes. |
| Jerarquía de operaciones | **De referencia** | Orden de operaciones y reglas de signos. |
| Guía de fracciones y porcentajes | **De referencia** | Procedimientos y errores típicos. |

Los tres **institucionales/pedagógicos** son los que hacen visible el valor del
RAG: contienen hechos que el modelo no puede tener. Los tres **de referencia**
contienen cosas que el modelo más o menos sabe — sirven para comprobar si el
contexto ayuda, estorba o da igual cuando la información ya está en los pesos.

> **Reemplácenlos por los documentos reales de su institución.** Los datos de
> abajo son verosímiles pero **ficticios**, construidos para que el experimento
> sea reproducible. Para S08 hacen falta 10–30 documentos reales con fuente y
> fecha.

In [ ]:
corpus = [
    {"id": "siee", "fuente": "Sistema Institucional de Evaluación (SIEE) 2026 — capítulo 3",
     "texto": (
        "La valoración del periodo académico en el área de matemáticas se compone de cuatro "
        "elementos con la siguiente ponderación: el examen final del periodo equivale al 40 por "
        "ciento de la nota, los talleres al 25 por ciento, los quices al 20 por ciento y el "
        "trabajo en clase al 15 por ciento. "
        "La escala institucional va de 1.0 a 5.0 y la nota mínima aprobatoria es 3.0. "
        "Cada estudiante dispone de dos oportunidades de recuperación por periodo; la nota "
        "máxima que puede obtenerse en una recuperación es 3.5. "
        "La inasistencia no justificada a un examen se califica con 1.0, salvo que el acudiente "
        "presente excusa dentro de los tres días hábiles siguientes.")},

    {"id": "plan_area", "fuente": "Plan de área de matemáticas 2026 — secuencia por grados",
     "texto": (
        "La secuencia del área organiza los contenidos así. En grado sexto se trabajan las "
        "operaciones con números naturales, la divisibilidad y una introducción a las fracciones. "
        "En grado séptimo se abordan los números enteros, la proporcionalidad directa e inversa y "
        "los porcentajes. "
        "En grado octavo se introducen las ecuaciones de primer grado con una incógnita, junto con "
        "el lenguaje algebraico y la traducción de enunciados verbales a expresiones matemáticas. "
        "En grado noveno se amplía a sistemas de dos ecuaciones lineales y a la función lineal. "
        "En grado décimo se trabajan la trigonometría y la geometría analítica. "
        "El área recomienda dedicar al menos dos sesiones a la traducción de enunciados verbales "
        "antes de introducir el algoritmo de despeje.")},

    {"id": "protocolo", "fuente": "Protocolo de errores frecuentes en matemáticas — 2026",
     "texto": (
        "Este protocolo recoge los errores más frecuentes detectados en las pruebas diagnósticas y "
        "la estrategia recomendada para cada uno. "
        "Descuentos sucesivos: los estudiantes suman los porcentajes, de modo que un 20 por ciento "
        "seguido de un 10 por ciento lo tratan como un 30 por ciento único. La estrategia "
        "recomendada es trabajar con el factor multiplicativo del precio que queda, es decir 0.80 y "
        "luego 0.90, organizados en una tabla de dos pasos, en lugar de sumar los porcentajes. "
        "Fracción del resto: cuando un enunciado dice 'un tercio de lo que quedaba', los estudiantes "
        "aplican la fracción al total original. Se recomienda pedir que escriban explícitamente "
        "cuánto queda antes de aplicar la segunda fracción. "
        "Proporcionalidad inversa: ante 'más obreros, menos días' aplican regla de tres directa. La "
        "estrategia es calcular primero el trabajo total en días-obrero. "
        "Redondeo contextual: al dividir personas entre vehículos o cajas entre camiones, el "
        "resultado debe redondearse hacia arriba porque no existen fracciones de vehículo.")},

    {"id": "formulario", "fuente": "Formulario de geometría — guía del estudiante",
     "texto": (
        "Áreas. Rectángulo: base por altura. Cuadrado: lado al cuadrado. Triángulo: base por altura "
        "dividido entre dos. Trapecio: la semisuma de las bases multiplicada por la altura. "
        "Círculo: pi por el radio al cuadrado. "
        "Perímetros. Rectángulo: dos por la suma de base y altura. Cuadrado: cuatro por el lado. "
        "Circunferencia: dos por pi por el radio. "
        "Volúmenes. Cubo: arista al cubo. Prisma rectangular: largo por ancho por alto. "
        "Cilindro: pi por el radio al cuadrado por la altura. "
        "Teorema de Pitágoras: en un triángulo rectángulo, el cuadrado de la hipotenusa es igual a "
        "la suma de los cuadrados de los catetos. "
        "Para figuras compuestas se descompone la figura en formas simples y se suman o restan sus "
        "áreas según corresponda.")},

    {"id": "jerarquia", "fuente": "Guía de operaciones — jerarquía y signos",
     "texto": (
        "El orden para resolver una expresión con varias operaciones es: primero los paréntesis, "
        "empezando por los más internos; después las potencias y raíces; luego las multiplicaciones "
        "y divisiones de izquierda a derecha; y por último las sumas y restas, también de izquierda "
        "a derecha. "
        "Un error común es resolver de izquierda a derecha sin respetar la jerarquía. "
        "Reglas de signos: al multiplicar o dividir, signos iguales dan positivo y signos distintos "
        "dan negativo. Restar un número negativo equivale a sumar su valor absoluto. "
        "La división entre cero no está definida: no existe ningún número que multiplicado por cero "
        "dé un resultado distinto de cero.")},

    {"id": "fracciones", "fuente": "Guía de fracciones, decimales y porcentajes",
     "texto": (
        "Para sumar o restar fracciones con distinto denominador se busca el mínimo común múltiplo y "
        "se convierten ambas al denominador común. Un error frecuente es sumar numeradores y "
        "denominadores por separado. "
        "Para multiplicar fracciones se multiplican numeradores entre sí y denominadores entre sí. "
        "Para dividir se multiplica por la fracción inversa. "
        "Calcular un porcentaje de una cantidad equivale a multiplicar por el porcentaje dividido "
        "entre cien. Para hallar el precio tras un aumento del quince por ciento se multiplica por "
        "1.15; para recuperar el precio anterior a partir del actual se DIVIDE entre 1.15, no se "
        "resta el quince por ciento. "
        "Una división entre un número menor que uno da un resultado mayor que el dividendo.")},
]

print(f"Documentos: {len(corpus)}")
print("Caracteres por documento:", [len(d["texto"]) for d in corpus])

---
## 6 · Chunking

El **chunk es la unidad de recuperación**: la búsqueda no encuentra documentos,
encuentra chunks. Si la respuesta a una pregunta queda partida entre dos, ningún
retrieval perfecto la traerá completa.

Usamos *fixed-size* con solapamiento, la estrategia más simple. Punto de partida
de la clase: 300–500 caracteres con 10–15% de overlap.

> **Experimenten:** bajen `CHUNK_SIZE` a 100 y súbanlo a 1500, reindexen y
> vuelvan a correr el Lab A. El chunking es una decisión de diseño **medible**,
> y con el harness de M2 pueden medirla de verdad en vez de opinar.

In [ ]:
CHUNK_SIZE = 400
OVERLAP    = 60


def partir_en_chunks(texto, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, inicio = [], 0
    while inicio < len(texto):
        chunks.append(texto[inicio:inicio + size])
        inicio += size - overlap
    return chunks


chunks, metadatos, ids_chunks = [], [], []
for doc in corpus:
    for j, ch in enumerate(partir_en_chunks(doc["texto"])):
        chunks.append(ch)
        metadatos.append({"fuente": doc["fuente"], "doc_id": doc["id"]})
        ids_chunks.append(f"{doc['id']}_chunk{j}")

print(f"{len(corpus)} documentos -> {len(chunks)} chunks")
print(f"Caracteres por chunk: min={min(len(c) for c in chunks)} max={max(len(c) for c in chunks)}")
print("\nEjemplo de chunk. Miren DÓNDE cortó: ¿partió una idea por la mitad?\n")
print(repr(chunks[2]))

---
## 7 · Lab A · Indexar y buscar: palabras clave vs. significado

Indexamos en Chroma y comparamos dos formas de buscar. La meta **no** es que la
semántica gane siempre, sino ver **cuándo gana, cuándo empata y cuándo trae
ruido**.

In [ ]:
import chromadb

cliente = chromadb.Client()
try:
    cliente.delete_collection("tutor_matematicas")
except Exception:
    pass
coleccion = cliente.create_collection("tutor_matematicas", metadata={"hnsw:space": "cosine"})

# FASE 1 · INDEXACIÓN (offline, una sola vez por corpus)
embeddings = st.encode(chunks, show_progress_bar=False)
coleccion.add(ids=ids_chunks, documents=chunks, metadatas=metadatos,
              embeddings=embeddings.tolist())
print("Indexados", coleccion.count(), "chunks en Chroma.")

In [ ]:
def buscar_keyword(consulta, k=3):
    """Baseline ingenuo: cuenta cuántas palabras de la consulta aparecen en el chunk."""
    palabras = set(consulta.lower().split())
    puntajes = [(sum(1 for p in palabras if p in ch.lower()), i) for i, ch in enumerate(chunks)]
    puntajes.sort(reverse=True)
    return [(s, ids_chunks[i], chunks[i]) for s, i in puntajes[:k]]


def buscar_semantica(consulta, k=3):
    """Búsqueda por significado: embebe la consulta y busca los vecinos más cercanos."""
    emb = st.encode([consulta]).tolist()
    r = coleccion.query(query_embeddings=emb, n_results=k)
    return [(round(1 - d, 3), i, doc, m["fuente"])
            for d, i, doc, m in zip(r["distances"][0], r["ids"][0],
                                    r["documents"][0], r["metadatas"][0])]


def comparar_busquedas(consulta, k=2):
    print("=" * 84)
    print("CONSULTA:", consulta)
    print("-" * 84)
    print("KEYWORD (coincidencias de palabras):")
    for s, cid, ch in buscar_keyword(consulta, k):
        print(f"  [{s} palabras] {cid:22s} {ch[:95]}...")
    print("SEMÁNTICA (similitud coseno):")
    for s, cid, ch, fu in buscar_semantica(consulta, k):
        print(f"  [sim {s:5.3f}]   {cid:22s} {ch[:95]}...")

In [ ]:
# 1 · LITERAL — las palabras de la consulta están en el texto. Las dos deberían acertar.
comparar_busquedas("¿Cuál es la nota mínima aprobatoria?")

In [ ]:
# 2 · COLOQUIAL — un acudiente preguntando lo mismo con otras palabras.
# Casi ninguna coincide con el documento: aquí el keyword se pierde.
comparar_busquedas("mi hijo sacó 2.8, ¿eso pasa o pierde la materia?")

In [ ]:
# 3 · DEL RUIDO — es del tema (evaluación) pero pregunta algo que NO está en el corpus.
# Noten que el top-k SIEMPRE devuelve k resultados, respondan o no la pregunta.
comparar_busquedas("¿cuántas horas de matemáticas hay a la semana en grado once?")

> **Qué observar en el Lab A**
>
> - **Consulta literal:** las dos búsquedas llegan al mismo chunk. Con palabras
>   compartidas, el keyword search es difícil de vencer — y es mucho más barato.
> - **Consulta coloquial:** "¿eso pasa o pierde la materia?" no comparte casi
>   nada con "la nota mínima aprobatoria es 3.0". El keyword se dispersa; la
>   semántica debería aterrizar en el chunk correcto. **Encontró por
>   significado, no por palabras.**
> - **Consulta del ruido:** el retrieval **siempre** devuelve k chunks, aunque
>   ninguno responda. Ordena por cercanía, no por "responde / no responde". Ese
>   ruido es el que se cuela al prompt en el Lab B, y por eso hace falta la
>   válvula de escape.

---
## 8 · Lab B · El RAG ingenuo, de punta a punta

### El prompt aumentado, adaptado a un tutor

La receta de clase tiene cuatro partes: instrucción, contexto, válvula de
escape y pregunta. La aplicamos con **una modificación necesaria**.

El prompt canónico dice *"responde SOLO con base en el contexto"*. Para un tutor
de matemáticas eso rompe la mitad de los casos: si le preguntan cuánto se paga
tras dos descuentos, la respuesta **no está en ningún documento** —hay que
calcularla— y el modelo se negaría a hacerlo.

La solución es un prompt de **dos modos** que el propio modelo distingue:

| Si la pregunta es… | El sistema debe… |
|---|---|
| un **dato institucional** (normas, plan de área, criterios) | responder solo desde el contexto y citar la fuente; si no está, admitirlo |
| un **problema de cálculo** | resolverlo paso a paso, usando el contexto solo si aporta una fórmula o estrategia |

La **válvula de escape** sigue siendo la línea más importante, pero acotada a
los datos institucionales: es ahí donde inventar es grave.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from pathlib import Path

MODELO_BASE   = "Qwen/Qwen2.5-1.5B-Instruct"
DIR_ADAPTADOR = "adaptadores/qwen-lora"      # el modelo afinado de M1
K = 3                                         # chunks recuperados por pregunta
MAX_TOKENS_GEN = 220

if not Path(DIR_ADAPTADOR).exists():
    raise FileNotFoundError(
        f"No se encuentra {DIR_ADAPTADOR}. Ejecuten S04_Lab_Fine_tuning_Qwen.ipynb "
        "o monten el Drive donde quedó guardado.")

tok = AutoTokenizer.from_pretrained(DIR_ADAPTADOR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

_base = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float16).to(DEVICE)
modelo = PeftModel.from_pretrained(_base, DIR_ADAPTADOR).eval()
print("Generador listo:", MODELO_BASE, "+ adaptador LoRA de M1")

In [ ]:
# Sin RAG: EXACTAMENTE el sistema evaluado en M2. Es el baseline de este notebook.
INSTRUCCION = (
    "Eres un tutor de matemáticas. Resuelve el siguiente problema explicando "
    "el procedimiento paso a paso y termina con la respuesta final."
)

# Con RAG: la instrucción de dos modos + la válvula de escape.
SYSTEM_RAG = (
    "Eres un tutor de matemáticas que dispone de documentos de consulta.\n"
    "- Si la pregunta pide un DATO de la institución (normas, plan de área, "
    "criterios de evaluación, protocolos), responde ÚNICAMENTE con base en el "
    "contexto y menciona la fuente. Si ese dato no aparece en el contexto, "
    'responde exactamente: "No tengo esa información en mis fuentes." '
    "No inventes datos institucionales.\n"
    "- Si la pregunta es un PROBLEMA matemático, resuélvelo explicando el "
    "procedimiento paso a paso y termina con la respuesta final. Usa el "
    "contexto solo si contiene una fórmula o estrategia que ayude.\n"
    "Sé breve y claro."
)


@torch.no_grad()
def _generar(system, user, max_new_tokens=MAX_TOKENS_GEN):
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    out = modelo.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def sistema_sin_rag(pregunta):
    return _generar(INSTRUCCION, pregunta)


def sistema_rag(pregunta, k=K, verbose=False):
    resultados = buscar_semantica(pregunta, k)                       # RETRIEVE
    contexto = "\n\n".join(f"[Fuente: {fu}]\n{doc}" for _, _, doc, fu in resultados)
    user = f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"        # AUGMENT
    if verbose:
        print("--- chunks recuperados ---")
        for s, cid, _, fu in resultados:
            print(f"  [sim {s:5.3f}] {cid:22s} ({fu[:46]}...)")
        print("--------------------------")
    return _generar(SYSTEM_RAG, user)                                # GENERATE

In [ ]:
# Comparación lado a lado sobre el bloque de CONOCIMIENTO: preguntas cuya
# respuesta vive en el corpus y en ningún otro sitio.
for caso in [c for c in eval_set if c["bloque"] == "conocimiento"][:2]:
    print("=" * 84)
    print("PREGUNTA:", caso["input"])
    print("ESPERADO:", caso["esperado"])
    print("\n--- SIN RAG (de memoria) ---")
    print(sistema_sin_rag(caso["input"])[:400])
    print("\n--- CON RAG (busca, lee y cita) ---")
    print(sistema_rag(caso["input"], verbose=True)[:400])
    print()

> **Qué esperar.** El dato ("el examen final vale 40%") es de *este* reglamento:
> no existe en la memoria del modelo. El sin-RAG debería responder algo genérico
> o —peor y más probable— **inventar un porcentaje con total seguridad**. El
> con-RAG debería dar el 40% citando el SIEE.
>
> Si el con-RAG falla, diagnostiquen con los tres modos de fallo de la clase:
> ¿llegó el chunk correcto al top-k (miren el `verbose`)? ¿llegó pero el modelo
> lo ignoró? ¿o el dato no estaba en el corpus?

In [ ]:
# Prueba ADVERSARIAL de retrieval: la respuesta NO está en el corpus.
# El sistema honesto lo admite; el deshonesto inventa.
p_adv = "¿Cuántas horas de matemáticas a la semana hay en grado once?"
print("PREGUNTA (sin respuesta en el corpus):", p_adv)
print("\n--- CON RAG ---")
print(sistema_rag(p_adv, verbose=True))
print("\n¿Dijo que no está en sus fuentes, o inventó un número?")

In [ ]:
# Y el otro modo: un PROBLEMA DE CÁLCULO. Aquí el contexto no trae la respuesta
# (hay que calcularla). Comprobamos que el prompt de dos modos no bloquea al
# tutor: debe resolver igual, no responder "no tengo esa información".
caso_calculo = next(c for c in eval_set if c["id"] == "dif-06")
print("PREGUNTA:", caso_calculo["input"])
print("\n--- CON RAG ---")
print(sistema_rag(caso_calculo["input"], verbose=True))
print(f"\nEsperado: 144000   |   ¿Correcta? "
      f"{respuesta_correcta(sistema_rag(caso_calculo['input']), '144000')}")

---
## 9 · El harness de M2, sin tocar

`sistema_rag` es una función `pregunta -> respuesta`: exactamente lo que el
harness de M2 recibe. Lo cargamos **idéntico**, porque si lo cambiáramos la
comparación de scorecards mediría la diferencia entre dos harness en vez del
efecto del RAG.

### Dimensión 3 · Aciertos de dominio — el criterio, caso por caso

Aquí nos apartamos deliberadamente de la plantilla del laboratorio de clase, y
conviene justificarlo porque es una decisión de diseño, no un descuido.

El lab define el acierto como `similitud >= 0.60 **o** juez >= 4`. Esa regla
funciona en dominios abiertos, pero **en matemáticas cuenta como acierto una
respuesta con el número equivocado**:

```
Referencia : "Respuesta final: 144000 pesos"
Respuesta  : "Respuesta final: 140000 pesos"     <- INCORRECTA
similitud de embeddings ≈ 0.99  ->  la regla del lab la daría por buena
```

Nuestro dominio tiene **verdad objetiva**, así que el criterio debe usarla.
Cada caso del eval set trae su propia regla en el campo `evaluacion`, versionada
junto al eval set:

| Tipo | Cómo se decide el acierto |
|---|---|
| `numerico` | El valor tras `Respuesta final:` coincide numéricamente con el esperado (tolerancia 1e-6; acepta fracciones). |
| `contiene_alguna` | La respuesta menciona alguna de las claves esperadas (p. ej. "no es primo"). |
| `abstencion` | La respuesta reconoce el límite de su alcance. |
| `prohibido` | Lista de textos que invalidan el caso: si aparecen, el sistema cayó en la trampa. |

Además del acierto registramos dos señales más, porque **fallar y mentir no son
lo mismo**:

- **`abstuvo`** — admitió no tener la información. No es un acierto, pero es el
  comportamiento correcto cuando el dato no está.
- **`alucino`** — ni acertó ni se abstuvo: afirmó algo incorrecto con seguridad.
  Es el fallo grave, y el que más importa reportar en un tutor.

In [ ]:
import re
import unicodedata


def _norm(texto):
    """Minúsculas y sin tildes: para comparar claves sin depender de la ortografía."""
    t = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in t if not unicodedata.combining(c))


# Marcadores de abstención honesta. Se detectan por separado del acierto porque
# "no lo sé" NO es una respuesta correcta, pero tampoco es una alucinación: es
# el comportamiento deseable cuando el sistema no tiene el dato.
#
# Cuidado con los marcadores genéricos: "falta" a secas produce falsos
# positivos constantes en este dominio ("faltan 1600 litros para llenarlo",
# "le hace falta recorrer 40 km"). Un falso positivo de abstención enmascara
# una alucinación, así que todos los marcadores son frases específicas.
MARCADORES_ABSTENCION = [
    "no tengo esa informacion", "no tengo informacion", "no dispongo",
    "no tengo acceso", "no puedo confirmar", "no aparece en", "no se especifica",
    "no se indica", "no se menciona", "no se proporciona", "no cuento con",
    "no esta en mis fuentes", "no tengo suficiente", "no lo se",
    "no puedo responder", "fuera de mi alcance", "no puedo ayudar",
    "no corresponde", "necesito mas datos", "necesito conocer",
    "falta un dato", "faltan datos", "falta informacion", "falta la distancia",
    "no es posible determinar", "no se puede determinar",
]


def se_abstuvo(respuesta):
    r = _norm(respuesta)
    return any(m in r for m in MARCADORES_ABSTENCION)


def evaluar_caso(caso, respuesta):
    """Dimensión 3: ¿esta respuesta cumple el criterio de ESTE caso?

    Por qué no usamos la regla genérica del lab de clase
    (`sim >= 0.60 or juez >= 4`): en matemáticas esa regla cuenta como acierto
    una respuesta con el número equivocado. "El resultado es 140000" y "el
    resultado es 144000" tienen similitud de embeddings ~0.99 y un juez pequeño
    les da la misma nota — pero una está mal. El criterio tiene que ser el del
    dominio, y aquí el dominio tiene verdad objetiva.

    Devuelve un dict con tres señales independientes:
      acierto  -> cumplió el criterio estricto del caso
      abstuvo  -> admitió no tener la información
      alucino  -> ni acertó ni se abstuvo (afirmó algo incorrecto con seguridad)
    """
    ev = caso["evaluacion"]
    r = _norm(respuesta)

    # 1) Texto prohibido: caer en la trampa invalida el caso de inmediato.
    prohibidos = [k for k in ev.get("prohibido", []) if _norm(k) in r]
    if prohibidos:
        return {"acierto": False, "abstuvo": False, "alucino": True,
                "motivo": f"cayó en la trampa: {prohibidos[0]!r}"}

    abstuvo = se_abstuvo(respuesta)
    tipo = ev["tipo"]

    if tipo.startswith("numerico"):
        acierto = respuesta_correcta(respuesta, ev["valor"])
        motivo = "número correcto" if acierto else f"esperaba {ev['valor']}, dio {valor_predicho(respuesta)}"
    elif tipo.startswith("contiene_alguna"):
        encontradas = [k for k in ev["claves"] if _norm(k) in r]
        acierto = bool(encontradas)
        motivo = f"contiene {encontradas[0]!r}" if acierto else "no menciona ninguna clave esperada"
    elif tipo == "abstencion":
        acierto = abstuvo or any(_norm(k) in r for k in ev.get("claves", []))
        motivo = "reconoció el límite" if acierto else "respondió como si estuviera en su dominio"
    else:
        raise ValueError(f"tipo de evaluación desconocido: {tipo}")

    # Alucinación = afirmó algo concreto y equivocado sin admitir que no sabía.
    alucino = (not acierto) and (not abstuvo)
    return {"acierto": acierto, "abstuvo": abstuvo, "alucino": alucino, "motivo": motivo}

### Dimensión 2 · LLM-as-a-judge

Un LLM lee la respuesta y la califica con una rúbrica 1–5. Da lo que ninguna
métrica automática da —juicio sobre si la explicación *enseña*— a escala de
máquina.

**La rúbrica es el producto, no el código.** Está versionada en la celda de
abajo y adaptada al dominio con dos reglas que no están en la plantilla de
clase:

1. **El resultado pesa más que la forma.** "Un procedimiento elegante con
   resultado equivocado no puede pasar de 3." Sin esa ancla, un juez pequeño
   premia la prosa bien estructurada aunque el número esté mal — que es
   exactamente el modo de fallo de nuestro modelo según M1.
2. **La longitud no es calidad.** Es la mitigación del sesgo de longitud (S06)
   escrita dentro de la propia rúbrica.

Usamos el modo **pointwise** para el scorecard (una nota por respuesta) y el
modo **pairwise** solo para el test de sesgo de posición.

> **Advertencia honesta, y vale para todo el notebook:** el juez es
> `Qwen2.5-1.5B-Instruct`, el mismo modelo base que estamos evaluando. Eso lo
> expone al **sesgo de auto-preferencia** documentado en S06. Lo declaramos
> aquí y lo cuantificamos en la sección de calibración: en nuestro dominio, a
> diferencia de casi cualquier otro, podemos comprobar si el juez tiene razón.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

JUEZ_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # si va lento: "Qwen/Qwen2.5-0.5B-Instruct"

juez_tok = AutoTokenizer.from_pretrained(JUEZ_MODEL)
juez_model = AutoModelForCausalLM.from_pretrained(
    JUEZ_MODEL, torch_dtype="auto").to(DEVICE).eval()

# ---------------------------------------------------------------------------
# LA RÚBRICA ES PARTE DE LA ENTREGA. Se versiona aquí, no se improvisa.
# Anclas explícitas por nivel (S06: "ancla cada nivel 1-5 con una descripción
# para que no juzgue a ojo") y control de longitud incorporado en el texto
# (S06: "pide concisión en la rúbrica").
# ---------------------------------------------------------------------------
RUBRICA = """Eres un evaluador de un tutor de matemáticas. Califica la RESPUESTA:

5 = El resultado final es CORRECTO y el procedimiento está bien explicado paso a paso.
4 = El resultado final es CORRECTO, pero la explicación omite un paso o tiene un detalle menor mejorable.
3 = El procedimiento es razonable pero el RESULTADO FINAL ES INCORRECTO, o no hay respuesta final.
2 = Hay errores conceptuales graves en el procedimiento.
1 = Incorrecta, inventada, o no responde lo que se preguntó.

Reglas:
- Lo que más pesa es que el RESULTADO sea correcto. Un procedimiento elegante con
  resultado equivocado no puede pasar de 3.
- La longitud no es calidad: una explicación más larga NO merece mejor nota.
- Si la pregunta no tiene respuesta posible (datos insuficientes, división entre
  cero, premisa falsa), la respuesta correcta es decirlo; inventar un número es 1."""


def _extraer_puntaje(texto):
    m = re.search(r"[1-5]", texto)
    return int(m.group()) if m else 3      # fallback neutro


def _generar_juez(system, user, max_new_tokens=6):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors="pt").to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    return juez_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)


def juez_puntua(pregunta, respuesta, esperada=None):
    """Modo POINTWISE: una nota de 1 a 5 para una respuesta. Es el que usa el harness."""
    ref = f"\n\nRespuesta de referencia (guía, no literal): {esperada}" if esperada else ""
    user = (f"{RUBRICA}\n\nPregunta: {pregunta}\nRespuesta a evaluar: {respuesta}{ref}\n\n"
            "Responde SOLO con un dígito del 1 al 5. Sin explicación.")
    return _extraer_puntaje(_generar_juez("Eres un evaluador estricto y objetivo.", user))


def juez_compara(pregunta, A, B):
    """Modo PAIRWISE: elige la mejor de dos. Se usa para cazar el sesgo de posición."""
    user = (f"Pregunta: {pregunta}\n\nRespuesta A: {A}\n\nRespuesta B: {B}\n\n"
            "¿Cuál respuesta es mejor? Responde SOLO con la letra A o B.")
    texto = _generar_juez("Eres un evaluador estricto y objetivo.", user, max_new_tokens=3).upper()
    m = re.search(r"[AB]", texto)
    return m.group() if m else "?"


def comparar_robusto(pregunta, X, Y):
    """Mitigación del sesgo de posición: evalúa en los dos órdenes y solo declara
    ganador si el veredicto coincide al invertir. Si se contradice, es empate."""
    v1 = juez_compara(pregunta, X, Y)   # X en la posición A
    v2 = juez_compara(pregunta, Y, X)   # X en la posición B
    if v1 == "A" and v2 == "B":
        return "X"
    if v1 == "B" and v2 == "A":
        return "Y"
    return "empate"

---
## El harness: las tres dimensiones en una función

`harness(eval_set, sistema)` recibe el eval set y **cualquier** función
`pregunta -> respuesta`, corre las tres dimensiones sobre cada caso y devuelve
el scorecard.

Que la firma sea tan genérica es deliberado: el mismo harness evaluará el
modelo de M1, el RAG de M3 y lo que venga después. Es la vara fija del
semestre, y solo sirve como vara si no cambia.

El scorecard desglosa los aciertos **por bloque**, porque el promedio global
mezcla tres preguntas distintas: ¿sabe calcular?, ¿conoce los datos de la
institución?, ¿es robusto ante trampas? Un sistema puede ser excelente en una y
pésimo en otra, y un número único lo ocultaría.

In [ ]:
import numpy as np


def harness(eval_set, sistema, nombre="sistema", verbose=True):
    """Las tres dimensiones sobre un sistema. `sistema` es cualquier función
    `pregunta -> respuesta`: el modelo de M1, el RAG de M3, lo que sea.

    Devuelve el scorecard: promedios por dimensión + el detalle caso por caso.
    """
    detalle = []
    for i, caso in enumerate(eval_set, 1):
        respuesta = sistema(caso["input"])

        sim = sim_embeddings(respuesta, caso["esperado"])        # Dimensión 1b
        pj = juez_puntua(caso["input"], respuesta, caso["esperado"])  # Dimensión 2
        ev = evaluar_caso(caso, respuesta)                       # Dimensión 3

        detalle.append({
            "id": caso["id"], "bloque": caso["bloque"], "subtipo": caso["subtipo"],
            "input": caso["input"], "respuesta": respuesta,
            "sim": round(sim, 3), "juez": pj,
            "acierto": ev["acierto"], "abstuvo": ev["abstuvo"], "alucino": ev["alucino"],
            "motivo": ev["motivo"],
            "formato_valido": formato_valido(respuesta),
            "palabras": len(respuesta.split()),
        })
        if verbose:
            marca = "OK " if ev["acierto"] else ("abs" if ev["abstuvo"] else "MAL")
            print(f"  [{i:2d}/{len(eval_set)}] {caso['id']:8s} {marca}  juez={pj}  sim={sim:.2f}  {ev['motivo'][:46]}")

    def prom(clave, filas=None):
        filas = filas if filas is not None else detalle
        return float(np.mean([d[clave] for d in filas])) if filas else 0.0

    calculo = [d for d in detalle if d["bloque"] == "calculo"]
    conocim = [d for d in detalle if d["bloque"] == "conocimiento"]
    advers  = [d for d in detalle if d["bloque"] == "adversarial"]

    return {
        "nombre": nombre,
        "n": len(detalle),
        # Dimensión 1
        "exactitud_calculo": prom("acierto", calculo),
        "sim_embeddings": prom("sim"),
        # Dimensión 2
        "juez_promedio": prom("juez"),
        # Dimensión 3
        "aciertos": sum(d["acierto"] for d in detalle),
        "aciertos_calculo": sum(d["acierto"] for d in calculo),
        "aciertos_conocimiento": sum(d["acierto"] for d in conocim),
        "aciertos_adversarial": sum(d["acierto"] for d in advers),
        "n_calculo": len(calculo), "n_conocimiento": len(conocim), "n_adversarial": len(advers),
        # Diagnóstico
        "alucinaciones": sum(d["alucino"] for d in detalle),
        "abstenciones": sum(d["abstuvo"] for d in detalle),
        "formato_valido": prom("formato_valido"),
        "palabras_media": prom("palabras"),
        "detalle": detalle,
    }


def imprimir_scorecard(*scorecards):
    """El scorecard: una tabla con el puntaje por dimensión. Acepta 1 o más
    sistemas para compararlos lado a lado sobre el mismo eval set."""
    nombres = [s["nombre"] for s in scorecards]
    ancho = 40
    total = ancho + 14 * len(scorecards)

    filas = [
        ("DIMENSIÓN 1 · MÉTRICA CLÁSICA", None, None),
        ("  1a · Exactitud en cálculo", lambda s: f"{s['exactitud_calculo']:.1%}", None),
        ("  1b · Similitud embeddings (0-1)", lambda s: f"{s['sim_embeddings']:.3f}", None),
        ("DIMENSIÓN 2 · LLM-AS-A-JUDGE", None, None),
        ("  2 · Nota media (1-5)", lambda s: f"{s['juez_promedio']:.2f}", None),
        ("DIMENSIÓN 3 · ACIERTOS DE DOMINIO", None, None),
        ("  3a · Total", lambda s: f"{s['aciertos']}/{s['n']}", None),
        ("  3b · Bloque cálculo", lambda s: f"{s['aciertos_calculo']}/{s['n_calculo']}", None),
        ("  3c · Bloque conocimiento", lambda s: f"{s['aciertos_conocimiento']}/{s['n_conocimiento']}", None),
        ("  3d · Bloque adversarial", lambda s: f"{s['aciertos_adversarial']}/{s['n_adversarial']}", None),
        ("DIAGNÓSTICO", None, None),
        ("  Alucinaciones (menos es mejor)", lambda s: str(s["alucinaciones"]), None),
        ("  Abstenciones honestas", lambda s: str(s["abstenciones"]), None),
        ("  Formato válido", lambda s: f"{s['formato_valido']:.0%}", None),
        ("  Palabras por respuesta", lambda s: f"{s['palabras_media']:.0f}", None),
    ]

    print("=" * total)
    print(f"{'':<{ancho}}" + "".join(f"{n[:13]:>14}" for n in nombres))
    print("-" * total)
    for etiqueta, fn, _ in filas:
        if fn is None:
            print(etiqueta)
        else:
            print(f"{etiqueta:<{ancho}}" + "".join(f"{fn(s):>14}" for s in scorecards))
    print("=" * total)


def guardar_scorecard(ruta, *scorecards):
    import csv
    nombres = [s["nombre"] for s in scorecards]
    filas = [
        ("exactitud_calculo", lambda s: round(s["exactitud_calculo"], 3)),
        ("sim_embeddings_prom", lambda s: round(s["sim_embeddings"], 3)),
        ("llm_juez_prom", lambda s: round(s["juez_promedio"], 3)),
        ("aciertos_total", lambda s: f"{s['aciertos']}/{s['n']}"),
        ("aciertos_calculo", lambda s: f"{s['aciertos_calculo']}/{s['n_calculo']}"),
        ("aciertos_conocimiento", lambda s: f"{s['aciertos_conocimiento']}/{s['n_conocimiento']}"),
        ("aciertos_adversarial", lambda s: f"{s['aciertos_adversarial']}/{s['n_adversarial']}"),
        ("alucinaciones", lambda s: s["alucinaciones"]),
        ("abstenciones", lambda s: s["abstenciones"]),
        ("formato_valido", lambda s: round(s["formato_valido"], 3)),
        ("palabras_media", lambda s: round(s["palabras_media"], 1)),
    ]
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["dimension"] + nombres)
        for clave, fn in filas:
            w.writerow([clave] + [fn(s) for s in scorecards])
    print(f"Guardado: {ruta}")


def tabla_por_subtipo(*scorecards):
    """Dónde falla el sistema, por tipo de dificultad. Es la tabla accionable:
    dice qué ejemplos añadir al corpus de entrenamiento."""
    nombres = [s["nombre"] for s in scorecards]
    ids = [d["id"] for d in scorecards[0]["detalle"]]
    print(f"{'caso':9s} {'subtipo':28s}" + "".join(f"{n[:12]:>13}" for n in nombres))
    print("-" * (37 + 13 * len(scorecards)))
    for i, cid in enumerate(ids):
        sub = scorecards[0]["detalle"][i]["subtipo"]
        celdas = ""
        for s in scorecards:
            d = s["detalle"][i]
            celdas += f"{('OK' if d['acierto'] else ('abst' if d['abstuvo'] else 'FALLA')):>13}"
        print(f"{cid:9s} {sub:28s}{celdas}")

### 9.1 · Las dos pasadas

Mismo eval set, mismo harness, mismas métricas. Lo único que cambia es si el
sistema consulta la biblioteca antes de responder.

In [ ]:
print("=== SIN RAG (el sistema de M2) ===")
sc_sin_rag = harness(eval_set, sistema_sin_rag, nombre="sin RAG")

In [ ]:
print("=== CON RAG ===")
sc_rag = harness(eval_set, lambda p: sistema_rag(p), nombre="con RAG")

---
## 10 · Comparación de scorecards

In [ ]:
imprimir_scorecard(sc_sin_rag, sc_rag)

In [ ]:
print("EL DESGLOSE QUE IMPORTA — RAG por bloque")
print("=" * 72)
for etiqueta, clave, n_clave in [
    ("Cálculo      (habilidad)", "aciertos_calculo", "n_calculo"),
    ("Conocimiento (RAG)      ", "aciertos_conocimiento", "n_conocimiento"),
    ("Adversarial  (robustez) ", "aciertos_adversarial", "n_adversarial"),
]:
    a, b = sc_sin_rag[clave], sc_rag[clave]
    n = sc_sin_rag[n_clave]
    flecha = "+" if b > a else ("=" if b == a else "-")
    print(f"  {etiqueta}  {a}/{n}  ->  {b}/{n}   [{flecha}{abs(b-a)}]")
print("=" * 72)
print()
print(f"Alucinaciones: {sc_sin_rag['alucinaciones']} -> {sc_rag['alucinaciones']}")
print(f"Abstenciones : {sc_sin_rag['abstenciones']} -> {sc_rag['abstenciones']}")
print()
print("La hipótesis de la sección 1 era: RAG gana en CONOCIMIENTO y no aporta")
print("(o estorba) en CÁLCULO. Comparen con lo que salió y repórtenlo tal cual.")

In [ ]:
import matplotlib.pyplot as plt

bloques = ["cálculo", "conocimiento", "adversarial"]
n_por_bloque = [sc_rag["n_calculo"], sc_rag["n_conocimiento"], sc_rag["n_adversarial"]]
sin = [sc_sin_rag["aciertos_calculo"] / sc_sin_rag["n_calculo"],
       sc_sin_rag["aciertos_conocimiento"] / sc_sin_rag["n_conocimiento"],
       sc_sin_rag["aciertos_adversarial"] / sc_sin_rag["n_adversarial"]]
con = [sc_rag["aciertos_calculo"] / sc_rag["n_calculo"],
       sc_rag["aciertos_conocimiento"] / sc_rag["n_conocimiento"],
       sc_rag["aciertos_adversarial"] / sc_rag["n_adversarial"]]

x = np.arange(3); ancho = 0.38
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(x - ancho/2, sin, ancho, label="Sin RAG (M2)")
ax.bar(x + ancho/2, con, ancho, label="Con RAG (M3)")
for i, (a, b) in enumerate(zip(sin, con)):
    ax.text(i - ancho/2, a + 0.02, f"{a:.0%}", ha="center", fontsize=9)
    ax.text(i + ancho/2, b + 0.02, f"{b:.0%}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f"{b}\n(n={n})" for b, n in zip(bloques, n_por_bloque)])
ax.set_ylim(0, 1.12); ax.set_ylabel("Aciertos de dominio")
ax.set_title("Efecto del RAG por bloque del eval set")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
tabla_por_subtipo(sc_sin_rag, sc_rag)

---
## 11 · Diagnóstico: ¿falló la búsqueda o falló la generación?

Cuando el RAG se equivoca hay **tres causas posibles** y se arreglan con cosas
distintas. La celda de abajo las separa automáticamente:

| Modo de fallo | Cómo se detecta | Se arregla con |
|---|---|---|
| **El dato no está en el corpus** | Ningún chunk lo contiene | Más documentos |
| **Fallo de retrieval** | El chunk correcto existe pero no entró al top-k | S08: hybrid search, reranking, query transformation |
| **Fallo de generación** | El chunk correcto llegó al prompt y el modelo lo ignoró | Prompt, modelo mayor, o fine-tuning sobre respuestas con contexto |

Distinguirlos importa porque S08 ataca **solo el segundo**. Si sus fallos son de
generación, las técnicas de la próxima semana no los van a arreglar.

In [ ]:
print("DIAGNÓSTICO DE LOS FALLOS DEL RAG")
print("=" * 84)

fallos = [d for d in sc_rag["detalle"] if not d["acierto"]]
if not fallos:
    print("Sin fallos en este eval set.")

for d in fallos:
    caso = next(c for c in eval_set if c["id"] == d["id"])
    recuperados = buscar_semantica(caso["input"], K)

    # ¿Alguna clave de la respuesta esperada aparece en algún chunk recuperado?
    claves = caso["evaluacion"].get("claves") or [caso["evaluacion"].get("valor", "")]
    claves = [k for k in claves if k]
    en_contexto = any(_norm(k) in _norm(doc) for k in claves for _, _, doc, _ in recuperados)
    en_corpus = any(_norm(k) in _norm(doc["texto"]) for k in claves for doc in corpus)

    if d["bloque"] == "calculo":
        causa = "CÁLCULO (el dato no vive en ningún documento: es habilidad, no conocimiento)"
    elif not en_corpus:
        causa = "EL DATO NO ESTÁ EN EL CORPUS -> añadir documentos"
    elif not en_contexto:
        causa = "FALLO DE RETRIEVAL -> el chunk existe pero no entró al top-k (S08)"
    else:
        causa = "FALLO DE GENERACIÓN -> el chunk correcto llegó y el modelo lo ignoró"

    print(f"[{d['id']} · {d['subtipo']}]  {causa}")
    print(f"   pregunta : {d['input'][:78]}")
    print(f"   motivo   : {d['motivo'][:78]}")
    print(f"   top-{K}    : {[cid for _, cid, _, _ in recuperados]}")
    print("-" * 84)

---
## 12 · Artefactos de la entrega

In [ ]:
import json
from pathlib import Path

Path("resultados").mkdir(exist_ok=True)

guardar_scorecard("resultados/scorecard_rag.csv", sc_sin_rag, sc_rag)

Path("resultados/m3_rag.json").write_text(json.dumps({
    "config": {"chunk_size": CHUNK_SIZE, "overlap": OVERLAP, "k": K,
               "n_documentos": len(corpus), "n_chunks": len(chunks),
               "modelo_embeddings": "paraphrase-multilingual-MiniLM-L12-v2",
               "generador": f"{MODELO_BASE} + LoRA (M1)"},
    "sin_rag": {k: v for k, v in sc_sin_rag.items() if k != "detalle"},
    "con_rag": {k: v for k, v in sc_rag.items() if k != "detalle"},
}, ensure_ascii=False, indent=2), encoding="utf-8")

# Las consultas donde el retrieval falló: el material de trabajo de S08.
fallidas = [{"id": d["id"], "input": d["input"], "subtipo": d["subtipo"],
             "motivo": d["motivo"],
             "top_k": [cid for _, cid, _, _ in buscar_semantica(d["input"], K)]}
            for d in sc_rag["detalle"] if not d["acierto"] and d["bloque"] != "calculo"]
Path("resultados/consultas_fallidas_s08.json").write_text(
    json.dumps(fallidas, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nConsultas fallidas guardadas para S08: {len(fallidas)}")
for p in sorted(Path("resultados").glob("*rag*")) + sorted(Path("resultados").glob("*s08*")):
    print(f"  {p}")

---
## 13 · Discusión

**1. ¿Dónde pagó el RAG?**
Si el bloque de conocimiento subió y el de cálculo se quedó igual, el resultado
confirma la tesis de S07: RAG da **conocimiento**, no **habilidad**. Es
exactamente lo que debía pasar y hay que reportarlo como confirmación, no como
decepción.

**2. ¿Bajó el bloque de cálculo?**
Es posible, y sería un hallazgo legítimo: meter tres chunks irrelevantes en el
prompt de un problema que solo requiere razonar añade ruido. Si pasó, la
mitigación natural es **enrutar**: usar RAG solo cuando la pregunta pide un dato
institucional. Conviene notar que es la misma conclusión a la que llegó el
pipeline BERT→Qwen de M1 —dar contexto que el modelo no necesitaba no ayudaba—,
solo que ahora con una causa distinta.

**3. ¿Bajaron las alucinaciones?**
Es la métrica que mejor mide la válvula de escape. Un tutor que dice "no tengo
esa información en mis fuentes" es más útil que uno que inventa el porcentaje
del examen final, aunque el acierto sea el mismo (cero) en ambos casos.

**4. ¿Qué tipo de fallo domina?**
De la sección 11. Si son de **retrieval**, S08 tiene las herramientas
(hybrid search, reranking, query transformation). Si son de **generación**, esas
técnicas no servirán y hay que mirar el prompt o el tamaño del modelo.

**5. El límite de siempre.**
21 casos, 4 de ellos de conocimiento. Un acierto vale 25 puntos porcentuales en
ese bloque. **Ninguna diferencia pequeña es concluyente**, y con este tamaño de
muestra lo honesto es hablar de dirección, no de magnitud.

---

### Lo que se llevan para S08

1. **Pipeline RAG completo y sin frameworks:** chunking → embeddings → Chroma →
   retrieve → augment → generate.
2. **`scorecard_rag.csv`** junto al de M2, medido con el mismo harness.
3. **El corpus indexado** — sustituyan los seis documentos ficticios por los
   reales de su institución (10–30 con fuente y fecha).
4. **`consultas_fallidas_s08.json`** — las consultas donde el retrieval falló.
   Sobre ellas se prueban hybrid search y reranking la próxima semana.